In [ ]:
print("Hello")

Hello


In [ ]:
# !pip install -q transformers datasets soundfile sentencepiece

In [ ]:
# !pip install -q transformers torch scipy

# from transformers import VitsModel, AutoTokenizer
# from IPython.display import Audio
# import torch

# model = VitsModel.from_pretrained("facebook/mms-tts-mar")
# tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-mar")

# text = "नमस्कार! हा आवाज गूगल कोलॅबमध्ये तयार झाला आहे."
# inputs = tokenizer(text, return_tensors="pt")

# with torch.no_grad():
#     output = model(**inputs).waveform

# Audio(output.numpy(), rate=model.config.sampling_rate)

KeyboardInterrupt: 

In [ ]:
# Do NOT reinstall/upgrade torch — Colab already ships a CUDA build; upgrading it breaks the GPU.
!pip install -q transformers accelerate scipy openai-whisper uroman bitsandbytes

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())   # must say True

import whisper, re
from transformers import AutoModelForCausalLM, AutoTokenizer, VitsModel, BitsAndBytesConfig
from IPython.display import Audio, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1) ASR — transcribe + auto-detect language
asr_model = whisper.load_model("small", device=DEVICE)

# 2) LLM — Qwen2.5-14B-Instruct, 4-bit quantized (~8GB) fits on Colab L4 (24GB) with headroom
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm_name = "Qwen/Qwen2.5-14B-Instruct"
llm_tok  = AutoTokenizer.from_pretrained(llm_name)
llm      = AutoModelForCausalLM.from_pretrained(llm_name, quantization_config=bnb_config, device_map="auto")

In [8]:
# Whisper uses ISO 639-1 ('mr'), MMS-TTS uses ISO 639-3 ('mar') — map between them
W2M   = {"mr":"mar","hi":"hin","gu":"guj","en":"eng","ta":"tam","te":"tel",
         "kn":"kan","bn":"ben","ml":"mal","pa":"pan","ur":"urd","or":"ory","as":"asm"}
NAMES = {"mr":"Marathi","hi":"Hindi","gu":"Gujarati","en":"English","ta":"Tamil",
         "te":"Telugu","kn":"Kannada","bn":"Bengali","ml":"Malayalam","pa":"Punjabi",
         "ur":"Urdu","or":"Odia","as":"Assamese"}

_tts = {}
def get_tts(code):
    mms = W2M.get(code, "eng")
    if mms not in _tts:
        _tts[mms] = (VitsModel.from_pretrained(f"facebook/mms-tts-{mms}"),
                     AutoTokenizer.from_pretrained(f"facebook/mms-tts-{mms}"))
    return _tts[mms]

In [8]:
from transformers import TextStreamer

def voice_chat(audio_path, max_new_tokens=1024):
    r = asr_model.transcribe(audio_path)
    code, query = r["language"], r["text"].strip()
    lang = NAMES.get(code, code)
    print(f"🗣  [{lang}] {query}\n🤖 ", end="")

    msgs = [{"role": "system", "content": f"You are a helpful assistant. The user may speak in {lang} or use English words mixed with {lang} (Hinglish/code-mix). Understand the intent and reply clearly in {lang}. Be concise."},
            {"role": "user", "content": query}]
    prompt = llm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = llm_tok([prompt], return_tensors="pt").to(llm.device)
    streamer = TextStreamer(llm_tok, skip_prompt=True, skip_special_tokens=True)
    with torch.no_grad():
        out = llm.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            streamer=streamer
        )
    reply = llm_tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

    m, t = get_tts(code)
    inputs = t(re.sub(r"[*#`_>]", "", reply), return_tensors="pt")
    with torch.no_grad():
        wav = m(**inputs).waveform
    return Audio(wav.numpy(), rate=m.config.sampling_rate)

In [ ]:
# import torch
# print("CUDA:", torch.cuda.is_available())          # must be True
# print("LLM on:", next(llm.parameters()).device)    # must be cuda:0

# from google.colab import files
# up = files.upload()                 # pick a .wav / .mp3 / .m4a / .webm
# voice_chat(next(iter(up)))

CUDA: False
LLM on: cpu


Saving hindi.mp3 to hindi (3).mp3


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🗣  [Hindi] नमस्ते, आप कैसे है? ये एक हिंदी परीक्षन रिकोडिंग है. क्रप्या मुझे भारत की राजदानी के बारे में बताये.
🤖 

In [9]:
import ipywidgets as widgets
import io

out = widgets.Output()
uploader = widgets.FileUpload(accept='.mp3,.wav,.m4a,.webm', multiple=False)
display(uploader, out)

def on_upload(change):
    if not uploader.value:
        return
    fname, fdata = next(iter(uploader.value.items()))
    out_path = f"/content/{fname}"
    with open(out_path, "wb") as f:
        f.write(fdata["content"])
    with out:
        out.clear_output()
        print(f"Saved: {out_path}\n")
        audio = voice_chat(out_path)   # prints 🗣 transcription + 🤖 LLM stream inside
        display(audio)

uploader.observe(on_upload, names="value")

FileUpload(value={}, accept='.mp3,.wav,.m4a,.webm', description='Upload')

Output()